# DataHub Error Case 03 — Ingestion Failure

**Monitor:** DataHub Ingestion Failure  
**Component:** datahub-gms

In [ ]:
import sys
sys.path.insert(0, '.')
import importlib
import datahub_test_utils
importlib.reload(datahub_test_utils)
from datahub_test_utils import *

r_login = get_token()
assert r_login.status_code == 200
session_cookie = r_login.cookies.get('actor')
headers = {'Cookie': f'actor={session_cookie}', 'Content-Type': 'application/json'}
print('Logged in')

In [ ]:
# Cell 2: Create ingestion source with bad recipe (unreachable MySQL)
bad_recipe = '{"source": {"type": "mysql", "config": {"host_port": "nonexistent-mysql:3306", "database": "bad", "username": "root", "password": "wrong"}}, "sink": {"type": "datahub-rest", "config": {"server": "http://datahub-datahub-gms:8080"}}}'

mutation = '''
    mutation {
        createIngestionSource(input: {
            name: "test-error-source",
            type: "mysql",
            config: {
                recipe: "%s",
                version: "0.13.3.2",
                executorId: "default"
            }
        })
    }
''' % bad_recipe.replace('"', '\\"').replace("\n", " ")

r = requests.post(f'{DATAHUB_URL}/api/graphql', headers=headers, json={"query": mutation})
print(f'Create source: {r.status_code}')
source_urn = r.json().get("data", {}).get("createIngestionSource")
print(f'Source URN: {source_urn}')

In [ ]:
# Cell 3: Trigger run
if source_urn:
    trigger = 'mutation { createIngestionExecutionRequest(input: { ingestionSourceUrn: "%s" }) }' % source_urn
    r2 = requests.post(f'{DATAHUB_URL}/api/graphql', headers=headers, json={'query': trigger})
    print(f'Trigger: {r2.status_code}')
    print(json.dumps(r2.json(), indent=2)[:300])

In [ ]:
# Cell 4: Find ingestion error logs
wait_for_logs(15)
hits = search_datahub_ingestion_errors(minutes_ago=3)
print_logs(hits)

all_hits = search_datahub_all_errors(minutes_ago=3)
print('--- All DataHub errors ---')
print_logs(all_hits)

In [ ]:
# Cell 5: Cleanup
if source_urn:
    delete = 'mutation { deleteIngestionSource(urn: "%s") }' % source_urn
    r3 = requests.post(f'{DATAHUB_URL}/api/graphql', headers=headers, json={'query': delete})
    print(f'Deleted: {r3.status_code}')

print('\n=== Summary ===')
print('Error:   Ingestion job failed')
print('Cause:   Unreachable source, bad recipe, wrong credentials')
print('Monitor: DataHub Ingestion Failure fires on ERROR count > 0 in 5 min')
print('Fix:     Check DataHub UI -> Ingestion -> run logs')